### ЗАДАЧА: Триаж обращений службы поддержки

Команда поддержки получает пакет строк с обращениями от разных сервисов.
Нужно обработать их так, чтобы:
- корректные обращения попали в итоговый список,
- проблемные записи не остановили весь пакет,
- по ошибкам собрался отдельный журнал,
- в конце было видно, в каких каналах остались не подтверждённые обращения,
- а также какова средняя длительность обработки по уровням приоритета.

Часть строк содержит ошибки в формате и числах,
часть использует неизвестный уровень приоритета или канал,
а часть передаёт неправильный флаг подтверждения.
        


In [4]:
# incident_id|service|severity|duration_min|channel|acknowledged
rows = [
    'INC-100|checkout|critical|12|pager|yes',
    'INC-101|search|high|7|slack|no',
    'INC-102|billing|medium|zero|email|yes',
    'INC-103|video|critical|-3|pager|no',
    'INC-104|feed|warning|5|slack|yes',
    'INC-105|auth|low|2|sms|no',
    'INC-106|cdn|high|4|email|maybe',
    'INC-107|ml|medium|9|slack|no',
]


class IncidentProcessingError(Exception):
    pass


class IncidentFormatError(IncidentProcessingError):
    pass


class SeverityError(IncidentProcessingError):
    pass


class DurationError(IncidentProcessingError):
    pass


class ChannelError(IncidentProcessingError):
    pass


class AcknowledgedFlagError(IncidentProcessingError):
    pass


def parse_incident(row):
    # TODO: split строку по '|'
    # TODO: убрать лишние пробелы у частей через strip()
    # TODO: ожидать 6 частей: incident_id, service, severity, duration_raw, channel, acknowledged_raw
    # TODO: если частей не 6 -> raise IncidentFormatError(...)
    # TODO: duration_raw преобразовать в float
    # TODO: при ошибке преобразования использовать raise DurationError(...) from exc
    # TODO: проверить, что duration > 0
    # TODO: проверить severity в {'low', 'medium', 'high', 'critical'}
    # TODO: проверить channel в {'email', 'slack', 'pager'}
    # TODO: проверить acknowledged_raw в {'yes', 'no'}
    # TODO: превратить acknowledged_raw в bool
    # TODO: вернуть словарь с разобранными полями
    arr = row.split('|')
    if len(arr) != 6:
        raise IncidentFormatError('частей должно быть 6!')
    incident_id, service, severity, duration_raw, channel, acknowledged_raw = arr[0].strip(), arr[1].strip(), arr[2].strip(), arr[3].strip(), arr[4].strip(), arr[5].strip()
    try:
        duration_raw = float(duration_raw)
    except ValueError as e:
        raise DurationError('некорректное значение duration!') from e
    if duration_raw <= 0:
        raise DurationError('duration должно быть больше 0!')
    if severity not in {'low', 'medium', 'high', 'critical'}:
        raise SeverityError('некорректное severity!')
    if channel not in {'email', 'slack', 'pager'}:
        raise ChannelError('некорректный channel!')
    if acknowledged_raw not in {'yes', 'no'}:
        raise AcknowledgedFlagError('некорректное acknowledged_raw!')
    if acknowledged_raw == 'yes':
        acknowledged_raw = True
    else:
        acknowledged_raw = False
    return {"incident_id": incident_id, "service": service, "severity": severity, "duration": duration_raw, "channel":channel, "acknowledged": acknowledged_raw}


def process_batch(rows):
    # TODO: создать списки incidents и errors
    # TODO: пройтись по rows циклом
    # TODO: внутри try вызвать parse_incident(row)
    # TODO: валидный incident добавить в incidents
    # TODO: IncidentProcessingError сохранить в errors как (row, error_type, message)
    # TODO: вернуть (incidents, errors)
    incidents = []
    errors = []
    for el in rows:
        try:
            obj = parse_incident(el)
            incidents.append(obj)
        except IncidentProcessingError as e:
            errors.append((el, type(e).__name__, e))
    return (incidents, errors)


# TODO: вызвать process_batch(rows)
# TODO: вывести количество валидных инцидентов и количество ошибок
# TODO: собрать error_counts: dict[str, int]
# TODO: собрать unacked_by_channel: dict[str, list[str]] только для acknowledged == False
# TODO: собрать average_duration_by_severity только по валидным строкам
# TODO: найти longest_incident среди валидных инцидентов по duration_min
# TODO: красиво вывести получившиеся структуры
res = process_batch(rows)
print(f'Количество валидных инцидентов: {len(res[0])}')
print(f'Количество ошибок: {len(res[1])}')
unacked_by_channel = {}
for inc in res[0]:
    if inc['acknowledged'] == False:
        unacked_by_channel.setdefault(inc['channel'], [])
        unacked_by_channel[inc['channel']].append(inc)
print(unacked_by_channel)
m = 0
longest_incident = None
for inc in res[0]:
    if inc['duration'] > m:
        m = inc['duration']
        longest_incident = inc
print(f'Самый долгий инцидент: {longest_incident}')

Количество валидных инцидентов: 3
Количество ошибок: 5
{'slack': [{'incident_id': 'INC-101', 'service': 'search', 'severity': 'high', 'duration': 7.0, 'channel': 'slack', 'acknowledged': False}, {'incident_id': 'INC-107', 'service': 'ml', 'severity': 'medium', 'duration': 9.0, 'channel': 'slack', 'acknowledged': False}]}
Самый долгий инцидент: {'incident_id': 'INC-100', 'service': 'checkout', 'severity': 'critical', 'duration': 12.0, 'channel': 'pager', 'acknowledged': True}
